# D1 PaddleOCR Test — D1 API (use_angle_cls)
PaddleOCR 2.7.0 — stable version used in D1. Tests same image as D1.

In [ ]:
# Cell 1 — Install D1-compatible versions
import subprocess, sys

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle-gpu==2.6.0'])
    print('✓ paddlepaddle-gpu 2.6.0')
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle==2.6.0'])
    print('⚠ CPU fallback: paddlepaddle 2.6.0')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddleocr==2.7.0'])
print('✓ paddleocr 2.7.0')

import paddle, paddleocr
print(f'PaddlePaddle: {paddle.__version__}')
print(f'PaddleOCR:    {paddleocr.__version__}')

In [ ]:
# Cell 2 — Init OCR using D1 API (use_angle_cls=True)
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang='en',
    use_angle_cls=True,
    use_gpu=True,
    show_log=False
)
print('✓ PaddleOCR D1 API ready')

In [ ]:
# Cell 3 — Upload and run OCR
from google.colab import files
print('Upload test image:')
uploaded = files.upload()

ocr_results = {}
for filename in uploaded.keys():
    print(f'\nRunning OCR: {filename}')
    result = ocr.ocr(filename, cls=True)
    ocr_results[filename] = result

    if not result or not result[0]:
        print('  No text detected')
    else:
        print(f'  {len(result[0])} regions detected:')
        for i, line in enumerate(result[0]):
            text = line[1][0]
            conf = line[1][1]
            print(f'  [{i+1:02d}] "{text}"  conf={conf:.3f}')

In [ ]:
# Cell 4 — Save results + notebook to Drive
import os, json, shutil, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Correct Drive path (no space in MyDrive)
drive_path = '/content/drive/MyDrive/AHFL-GPU-Tests/ahfl-working-Gpu'

if not os.path.exists(drive_path):
    raise FileNotFoundError(f'Drive path not found: {drive_path}\nVerify folder exists on Drive first.')

print(f'✓ Drive path: {drive_path}')

# Save OCR results as JSON
out = {}
for filename, result in ocr_results.items():
    out[filename] = [
        {'text': line[1][0], 'conf': round(line[1][1], 4)}
        for line in (result[0] or [])
    ]

json_path = os.path.join(drive_path, 'd1_ocr_results.json')
with open(json_path, 'w') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

sz = os.path.getsize(json_path)
print(f'✓ d1_ocr_results.json ({sz} bytes)')

# Copy this notebook
nb_src = '/content/test_d1_ocr.ipynb'
if os.path.exists(nb_src):
    shutil.copy2(nb_src, os.path.join(drive_path, 'test_d1_ocr.ipynb'))
    print('✓ test_d1_ocr.ipynb')

# Verify — list files on Drive
print(f'\nFiles on Drive:')
for f in sorted(os.listdir(drive_path)):
    fpath = os.path.join(drive_path, f)
    if os.path.isfile(fpath):
        print(f'  • {f} ({os.path.getsize(fpath)} bytes)')

print('\n✓ Done')